In [ ]:
import pandas as pd

ENDPOINT_CSV_URL = "https://raw.githubusercontent.com/dadirad/blue-team-lab/main/files/endpoint_events.csv"
df = pd.read_csv(ENDPOINT_CSV_URL)
df["ts"] = pd.to_datetime(df["ts"], utc=True, errors="coerce")
df.head()


# Blue Team Lab (Endpoint-centric)
## 60-minute investigation using endpoint telemetry

### Scenario
You are a blue team analyst reviewing endpoint telemetry: process creation, network connections, file writes, and persistence.

### Goals
- Identify the suspicious execution chain
- Identify the artifact and persistence mechanism
- Identify likely C2 destination
- Recommend containment and one detection improvement
- Document your findings clearly

### Deliverables
1) Answers to the 8 investigation questions
2) A 6–10 sentence incident summary


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("endpoint_events.csv")
df["ts"] = pd.to_datetime(df["ts"], utc=True, errors="coerce")
df = df.sort_values("ts").reset_index(drop=True)
df.head(10)


## Task 0 (2 minutes): Confirm what data you have
Run the next cell to see which endpoint event types exist.


In [ ]:
df["event_type"].value_counts(dropna=False)


## Task 1 (10 minutes): Identify suspicious parent → child process chains
Classic suspicious patterns include:
- outlook.exe → powershell.exe
- winword.exe/excel.exe → powershell.exe/cmd.exe/wscript.exe
- Encoded commands (-enc), hidden windows (-w hidden), no profile (-nop)

Run the next cell and capture your primary lead.


In [ ]:
proc = df[df["event_type"]=="PROCESS_CREATE"].copy()
proc["process_l"] = proc["process"].fillna("").str.lower()
proc["parent_l"] = proc["parent_process"].fillna("").str.lower()
proc["cmd_l"] = proc["cmdline"].fillna("").str.lower()

susp_chain = proc[
    (proc["parent_l"].str.contains("outlook|winword|excel|chrome|acrord32")) &
    (proc["process_l"].str.contains("powershell|cmd|wscript|cscript|mshta|rundll32"))
].copy()

susp_chain[["ts","host","user","parent_process","process","cmdline","notes"]].sort_values("ts")


## Task 2 (8 minutes): Look for encoded PowerShell or suspicious flags
Run the next cell and identify any PowerShell commands using -enc or other suspicious flags.


In [ ]:
ps = proc[proc["process_l"].eq("powershell.exe")].copy()

encoded_or_hidden = ps[
    ps["cmd_l"].str.contains(r" -enc | -encodedcommand | -nop | -w hidden | -windowstyle hidden ", regex=True)
].copy()

encoded_or_hidden[["ts","host","user","parent_process","process","cmdline","notes"]].sort_values("ts")


## Task 3 (12 minutes): Pivot to a single host timeline
1) Set HOST to the host you believe is compromised
2) Run the next cell to see the full sequence for that host
3) Identify the chain: execution → network → file write → execution → persistence → repeat network


In [ ]:
HOST = ""  # e.g., "WS-023"
if not HOST:
    print("Set HOST to a value like 'WS-023' and re-run this cell.")
else:
    focus = df[df["host"]==HOST].copy()
    cols = ["ts","event_type","host","user","process","parent_process","cmdline","hash","network_dst","network_port","action","notes"]
    display(focus[cols].sort_values("ts"))


## Task 4 (8 minutes): Identify artifacts (file writes) and persistence
Capture:
- the file path written (artifact)
- any hash value
- the persistence mechanism (Run key, scheduled task, service, etc.)


In [ ]:
file_writes = df[df["event_type"]=="FILE_WRITE"].copy()
persist = df[df["event_type"].str.contains("PERSIST", na=False)].copy()

display(file_writes[["ts","host","user","process","cmdline","hash","notes"]].sort_values("ts"))
display(persist[["ts","host","user","event_type","process","cmdline","notes"]].sort_values("ts"))


## Task 5 (8 minutes): Identify likely C2 destination and beaconing
Repeated outbound connections to the same destination at a regular interval can be C2 beaconing.
Run the next cell and identify the most suspicious destination.


In [ ]:
net = df[df["event_type"]=="NETWORK_CONNECT"].copy()
net = net.sort_values(["host","process","network_dst","network_port","ts"])

net["delta_sec"] = net.groupby(["host","process","network_dst","network_port"])["ts"].diff().dt.total_seconds()

net_stats = (
    net.groupby(["host","process","network_dst","network_port"], dropna=False)
       .agg(
           count=("ts","count"),
           mean_delta=("delta_sec","mean"),
           std_delta=("delta_sec","std"),
           first=("ts","min"),
           last=("ts","max")
       )
       .reset_index()
       .sort_values(["count","std_delta"], ascending=[False, True])
)

net_stats.head(25)


## Submit: 8 Investigation Questions
1) Suspected compromised host:
2) Suspicious process chain (parent → child) OR suspicious artifact:
3) First suspicious timestamp (with evidence):
4) Primary lead (what triggered your investigation?):
5) Evidence that supports malicious activity (2–3 bullets):
6) Best-guess attack technique (plain language is fine):
7) Immediate containment actions (1–2 actions):
8) One detection improvement idea:

## Incident Summary (6–10 sentences)
**Title:** Suspected endpoint compromise on `<host>`

- At `<time>`, `<parent process>` spawned `<child process>` with suspicious command-line behavior.
- The activity indicates `<download/execution>` of `<artifact>` (include hash if present).
- Persistence was established via `<mechanism>`.
- The host initiated repeated outbound connections to `<destination>` suggesting possible command and control.
- Confidence: `<high/medium/low>` because `<reason>`.
- Recommended containment: `<actions>`.
- Detection improvement: `<specific detection rule/tuning idea>`.
